# 02: Compare four rules with fixed oracle utility

This synthetic key-door example compares FIFO, LRU, Utility and Evidence per byte policy objects with supplied utility values: the key gets 1.0 and distractors get 0.1. It retrieves the surviving key at the final decision, without an outcome-driven utility update or measured record-removal effects.

The notebook produces success and retention tables, plots and paired tests across its configured budgets and seeds. Those quantities describe this oracle workload. The [laboratory comparison](../docs/09-Tasks-Benchmarks.md) runs the four native tasks with learned utility, explicit importance, the complete control set and measured retention costs. The [saved experiments](README.md#read-the-saved-experiments) and [live-pruning comparisons](README.md#select-a-live-prune-with-an-episode-replay) retain their separate protocols and outcomes.

Demo and paper modes choose sample sizes; neither establishes that this example tests the broader memory-policy hypothesis.


## 1. Setup

In [ ]:
RUN_MODE = "demo"  # @param ["demo", "paper"]

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "notebooks":
    NOTEBOOK_DIR = NOTEBOOK_DIR.parent
sys.path.insert(0, str(NOTEBOOK_DIR))

from _utils import (
    NotebookConfig, print_config, print_repro_stamp, set_all_seeds,
    RunManager, paired_permutation_test, bootstrap_ci, effect_size,
    setup_plotting, add_caption, COLORS, POLICY_COLORS, plot_pareto_frontier,
)

from reckon.core.resources import MemoryBudget
from reckon.memory import CausalMemoryManager, PayloadType
from reckon.memory import FIFOPolicy, LRUPolicy, UtilityPolicy, EvidencePerBytePolicy

import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
import random

setup_plotting()
print("Imports OK!")

In [ ]:
config = NotebookConfig(
    notebook_name="02_policy_bakeoff",
    run_mode=RUN_MODE,
    seed=42,
    seed_count=5 if RUN_MODE == "paper" else 2,
    budget_slots=50,
    task_name="keydoor",
    num_episodes=50 if RUN_MODE == "paper" else 10,
    policy_name="all",
    evidence_weight=0.5,
)
print_config(config)
print_repro_stamp(config)
set_all_seeds(config.seed)

## 2. What this comparison can distinguish

FIFO removes older records. LRU removes records used less recently, although this fixture does not read the key until the final decision. Utility ranks the supplied score per byte. Evidence per byte combines that score with an available measured effect or proxy, also per byte.

This fixture supplies neither measured effects nor proxies. Utility and Evidence per byte can therefore rank the records alike; a shared outcome cannot establish the value of measuring record effects. The comparison asks how age, access history and the supplied oracle score affect survival of the key under its declared budgets. The broader question of whether measurement improves retention is tested by the separate [measured comparisons](README.md#select-a-live-prune-with-an-episode-replay).

The selected policy orders removals for byte, slot and token pressure together; see the [memory manager design](../docs/08-Causal-Memory-Manager-Design.md).


## 3. Minimal Theory

The Python `EvidencePerBytePolicy` uses a utility/available-signal score divided by byte cost:

$$\mathrm{priority}(r)=\frac{(1-w)u(r)+w\,s(r)}{\max(c_{\mathrm{bytes}}(r),1)}$$

`s` takes the positive part of the first present ablation or counterfactual measurement. Proxy magnitude applies only when neither measurement exists; absent scores contribute zero. A harmful or zero measurement cannot select a proxy instead. This experiment supplies oracle utility without those signals, so it compares retention rules without measuring the benefit of effect-based retention. The utility-based rankings can coincide. Python calls the kernel’s `EvidencePerByte` rule, described in the [memory manager design](../docs/08-Causal-Memory-Manager-Design.md).

See [Math Core](../docs/03-Math-Core.md) for the broader research framing.


## 4. System Mapping

```mermaid
flowchart LR
    subgraph TASK["KeyDoor Task"]
        T1["Generate episodes with\nkey observation at step K"]
    end

    subgraph POLICIES["Retention Policies"]
        P1[FIFO]
        P2[LRU]
        P3[Utility]
        P4[Evidence per byte]
    end

    subgraph CMM["CausalMemoryManager"]
        M1["write() records"]
        M2["prune() when over budget"]
        M3["retrieve() at decision time"]
    end

    subgraph METRICS["Evaluation"]
        E1["Success rate"]
        E2["Memory efficiency"]
        E3["Statistical tests"]
    end

    TASK --> CMM
    POLICIES --> CMM
    CMM --> METRICS
```


## 5. Experiment Design

**Unit of analysis:** One episode (success/failure)

**Independent variables:**
- Policy: FIFO, LRU, Utility, Evidence per byte
- Budget: 5, 10, 15, 20, 30 slots

**Dependent variables:**
- Success rate (primary)
- Key retention rate (secondary)

**Seeds:** 5 seeds × 50 episodes = 250 per condition (paper mode)

**Statistical tests:**
- Paired permutation test (α = 0.05) comparing Evidence per byte vs each baseline
- Bootstrap 95% CI for all success rates

## 6. Demo Run

In [ ]:
def run_keydoor_episode(policy, budget_slots: int, seed: int, episode_length: int = 20) -> dict:
    """
    Run one KeyDoor episode.
    
    The agent sees a key at a random early step, then must remember it
    when reaching the door at the final step.
    """
    random.seed(seed)
    np.random.seed(seed)
    
    budget = MemoryBudget(max_slots=budget_slots, max_bytes=budget_slots * 100)
    cmm = CausalMemoryManager(budget, policy, f"keydoor-{seed}")
    
    # Key appears at random step in first half
    key_step = random.randint(2, episode_length // 2)
    key_color = random.choice(["red", "blue", "green", "yellow"])
    key_record_id = None
    
    # Simulate episode
    for step in range(episode_length):
        cmm.set_step(step)
        
        if step == key_step:
            # KEY observation (important!)
            key_record_id = cmm.write(
                payload={"key": key_color, "room": "key_room"},
                payload_type=PayloadType.CUE,
                tags=["key", key_color, "important"],
                initial_utility=1.0,
            )
        else:
            # Distractor observation
            cmm.write(
                payload={"room": f"corridor_{step}", "objects": []},
                payload_type=PayloadType.OBSERVATION,
                tags=["observation", "distractor"],
                initial_utility=0.1,
            )
    
    # Decision time: can we retrieve the key?
    hits = cmm.retrieve(query={"tags": ["key"]}, limit=1)
    key_retrieved = len(hits) > 0 and key_color in hits[0].tags
    
    # Check if key record still exists
    key_retained = key_record_id in [r.id for r in cmm.get_all_records()]
    
    return {
        "success": key_retrieved,
        "key_retained": key_retained,
        "key_step": key_step,
        "final_slots": cmm.current_usage_slots,
        "seed": seed,
    }

# Quick sanity check
result = run_keydoor_episode(EvidencePerBytePolicy(0.5), budget_slots=10, seed=42)
print(f"Demo episode: success={result['success']}, key_retained={result['key_retained']}")

## 7. Full Run

In [ ]:
# Define policies and budgets
POLICIES = {
    "FIFO": FIFOPolicy(),
    "LRU": LRUPolicy(),
    "Utility": UtilityPolicy(),
    "Evidence per byte": EvidencePerBytePolicy(config.evidence_weight),
}

BUDGETS = [5, 10, 15, 20, 30] if RUN_MODE == "paper" else [5, 10, 20]

# Run experiments
results = defaultdict(lambda: defaultdict(list))

total_runs = len(POLICIES) * len(BUDGETS) * config.seed_count * config.num_episodes
print(f"Running {total_runs} episodes...")

run_count = 0
for policy_name, policy in POLICIES.items():
    for budget in BUDGETS:
        for seed_offset in range(config.seed_count):
            base_seed = config.seed + seed_offset * 1000
            
            for ep in range(config.num_episodes):
                seed = base_seed + ep
                result = run_keydoor_episode(policy, budget, seed)
                results[(policy_name, budget)]["success"].append(result["success"])
                results[(policy_name, budget)]["key_retained"].append(result["key_retained"])
                run_count += 1
                
                if run_count % 100 == 0:
                    print(f"  {run_count}/{total_runs}...")

print(f"Done! {run_count} episodes completed.")

## 8. Results

In [ ]:
# Compute success rates with CIs
summary = []

for policy_name in POLICIES.keys():
    for budget in BUDGETS:
        successes = results[(policy_name, budget)]["success"]
        rate = np.mean(successes)
        ci_low, ci_high = bootstrap_ci(successes, confidence=0.95)
        
        summary.append({
            "policy": policy_name,
            "budget": budget,
            "success_rate": rate,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "n": len(successes),
        })

# Print table
print("Success Rates by Policy and Budget:")
print("=" * 70)
print(f"{'Policy':<10} {'Budget':<8} {'Rate':<10} {'95% CI':<20} {'N':<6}")
print("-" * 70)
for row in summary:
    ci_str = f"[{row['ci_low']:.3f}, {row['ci_high']:.3f}]"
    print(f"{row['policy']:<10} {row['budget']:<8} {row['success_rate']:.3f}      {ci_str:<20} {row['n']:<6}")

In [ ]:
# Statistical tests: Evidence per byte vs each baseline
print("\nStatistical Tests (Evidence per byte vs Baselines):")
print("=" * 60)

for budget in BUDGETS:
    print(f"\nBudget = {budget} slots:")
    evidence_results = results[("Evidence per byte", budget)]["success"]
    
    for baseline in ["FIFO", "LRU", "Utility"]:
        baseline_results = results[(baseline, budget)]["success"]
        
        # Paired permutation test
        p_value = paired_permutation_test(
            baseline_results, evidence_results,
            n_permutations=1000, seed=config.seed
        )
        
        # Effect size
        es = effect_size(baseline_results, evidence_results)
        
        sig = "*" if p_value < 0.05 else ""
        print(f"  Evidence per byte vs {baseline:<8}: p={p_value:.4f}{sig}, d={es:.3f}")

In [ ]:
# Plot: Success rate by budget for each policy
fig, ax = plt.subplots(figsize=(10, 6))

for i, policy_name in enumerate(POLICIES.keys()):
    budgets_arr = []
    rates = []
    ci_lows = []
    ci_highs = []

    for row in summary:
        if row["policy"] == policy_name:
            budgets_arr.append(row["budget"])
            rates.append(row["success_rate"])
            ci_lows.append(row["ci_low"])
            ci_highs.append(row["ci_high"])

    color = list(COLORS.values())[i]
    ax.plot(budgets_arr, rates, 'o-', label=policy_name, color=color, linewidth=2, markersize=8)
    ax.fill_between(budgets_arr, ci_lows, ci_highs, alpha=0.2, color=color)

ax.set_xlabel("Memory Budget (slots)")
ax.set_ylabel("Success Rate")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower right")
ax.axhline(1.0, color="gray", linestyle="--", alpha=0.3)

n_total = sum(row["n"] for row in summary) // len(POLICIES)
seed_ranges = [
    f"{config.seed + offset * 1000}-{config.seed + offset * 1000 + config.num_episodes - 1}"
    for offset in range(config.seed_count)
]
add_caption(ax, "Observed synthetic success rates by policy and budget",
            n=n_total, seeds="; ".join(seed_ranges))

plt.tight_layout()
plt.show()

In [ ]:
# Pareto frontier: Success rate vs Memory used
pareto_data = []
for row in summary:
    pareto_data.append({
        "x": row["budget"],  # Memory cost
        "y": row["success_rate"],  # Performance
        "label": f"{row['policy']}@{row['budget']}",
    })

fig, ax = plt.subplots(figsize=(10, 6))
plot_pareto_frontier(
    ax,
    [d["x"] for d in pareto_data],
    [d["y"] for d in pareto_data],
    labels=[d["label"] for d in pareto_data],
    xlabel="Memory Budget (slots)",
    ylabel="Success Rate",
)
add_caption(ax, "Synthetic success versus configured slot budget", n=len(pareto_data))
plt.tight_layout()
plt.show()

### Failure Analysis

First inspect the worst observed summary entry. The separate trace then searches twenty seeds for a FIFO failure at five slots and reports whether it found one. Match the lost key to its prune event before explaining the failure.


In [ ]:
# Find the worst (policy, budget) combination
worst_config = min(summary, key=lambda x: x["success_rate"])
print(f"Worst configuration: {worst_config['policy']} at budget={worst_config['budget']}")
print(f"Success rate: {worst_config['success_rate']:.1%}")

# Run a detailed failure trace
def trace_failure(policy, budget_slots: int, seed: int, episode_length: int = 20):
    """Run one episode with detailed event logging."""
    random.seed(seed)
    np.random.seed(seed)

    budget = MemoryBudget(max_slots=budget_slots, max_bytes=budget_slots * 100)
    cmm = CausalMemoryManager(budget, policy, f"trace-{seed}")

    key_step = random.randint(2, episode_length // 2)
    key_color = random.choice(["red", "blue", "green"])
    key_id = None

    for step in range(episode_length):
        cmm.set_step(step)
        if step == key_step:
            key_id = cmm.write(
                payload={"key": key_color},
                payload_type=PayloadType.CUE,
                tags=["key", key_color],
                initial_utility=1.0,
            )
        else:
            cmm.write(
                payload={"room": f"corridor_{step}"},
                payload_type=PayloadType.OBSERVATION,
                tags=["observation"],
                initial_utility=0.1,
            )

    # Analyze events
    events = cmm.get_event_log()
    prune_events = [e for e in events if e.get('op_type') == 'prune']
    key_prune = next((e for e in prune_events if e.get('record_id') == key_id), None)

    hits = cmm.retrieve(query={"tags": ["key"]}, limit=1)
    success = len(hits) > 0

    return {
        "key_step": key_step,
        "key_pruned_at": key_prune.get('timestamp') if key_prune else None,
        "success": success,
        "total_prunes": len(prune_events),
        "key_color": key_color,
    }

# Find a seed where FIFO fails at budget=5
failure_seed = None
for s in range(config.seed, config.seed + 20):
    trace = trace_failure(FIFOPolicy(), budget_slots=5, seed=s)
    if not trace["success"]:
        failure_seed = s
        break

if failure_seed is not None:
    trace = trace_failure(FIFOPolicy(), budget_slots=5, seed=failure_seed)
    print(f"\nFailure trace (seed={failure_seed}):")
    print(f"  At step {trace['key_step']}, the system wrote the key ({trace['key_color']}).")
    print(f"  At step {trace['key_pruned_at']}, budget pressure hit.")
    print(f"  The trace lost the key; inspect the actual prune reason and selection path.")
    print(f"  Game over. The agent cannot succeed.")
else:
    print("\nNo failure found in first 20 seeds (try lower budget).")

The trace summarizes actual prune events using `op_type` and logical `timestamp`. It does not export those events or create a decision-node dependency graph. A high oracle utility score is a supplied signal, not measured causal influence. FIFO still removes by age; the selected order applies to every storage dimension.


## 9. Debugger Walkthrough

This notebook currently exports comparison metrics and figures, not an event-log run. Inspect those tables/figures here; there is no generated log to import merely because a run directory exists. For an explicitly saved event log from another example, use the [native debugger quick start](../src/reckon/viz/README.md#quick-start). No ablation values or pair-bound graph edges are produced by this comparison.

## 10. Scope and related comparisons

This is a synthetic key-door fixture with manually assigned oracle utility. It supplies no ablation scores and makes no outcome-driven utility update. Its statistics describe those choices; they do not establish a causal-retention advantage or a published benchmark. The manager refuses an incoming record larger than its byte or token allowance before pruning existing records.

Use [notebook 03](03_contract_evaluation_executable.ipynb) for a measured record removal, [notebook 05](05_rl_env_adapter_minigrid.ipynb) for the live MiniGrid memory path, and the [configured native comparison](../docs/09-Tasks-Benchmarks.md) for learned utility and the full control set. [Notebook 07](07_scaling_and_perf_envelope.ipynb) examines synthetic store scaling; it does not extend this task's horizon. The [notebook guide](README.md) identifies the completed comparisons, their original sources and remaining workload limits.


## Export Artifacts

In [ ]:
run = RunManager(config)
run.save_config()

# Save metrics
run.save_metrics(summary)

# Save figures
fig, ax = plt.subplots(figsize=(10, 6))
for i, policy_name in enumerate(POLICIES.keys()):
    budgets_arr = [row["budget"] for row in summary if row["policy"] == policy_name]
    rates = [row["success_rate"] for row in summary if row["policy"] == policy_name]
    ax.plot(budgets_arr, rates, 'o-', label=policy_name, color=list(COLORS.values())[i])
ax.set_xlabel("Memory Budget (slots)")
ax.set_ylabel("Success Rate")
ax.legend()
run.save_figure(fig, "policy_comparison")
plt.close(fig)

# Write summary
best_evidence = max([row["success_rate"] for row in summary if row["policy"] == "Evidence per byte"])
best_fifo = max([row["success_rate"] for row in summary if row["policy"] == "FIFO"])
run.write_summary({
    "best_evidence_rate": best_evidence,
    "best_fifo_rate": best_fifo,
    "improvement": best_evidence - best_fifo,
    "total_episodes": sum(row["n"] for row in summary),
})

run.print_artifacts()